In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
from PIL import Image
import os
import shutil

# 1. Start with the base model
model = models.resnet18(weights=None) # No need to download ImageNet weights again

# 2. Modify the head exactly like before (during training)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) 

# 3. Setup the device (M2)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

In [2]:


# 4. Load the saved state dictionary
model_path = 'tripos_classifier_v4.pth'
model.load_state_dict(torch.load(model_path, map_location=device))

# 5. Set to Evaluation Mode
model.eval() 
print("Model loaded")



Model loaded


In [6]:

# 2. Setup paths
unlabeled_folder = '/Users/aishago/Downloads/output_pngs/D20250617T081548_IFCB202'
destination_folder = './potential_tripos_hits-1'
os.makedirs(destination_folder, exist_ok=True)

# 3. Prediction Loop
# Use the same transforms as your validation set
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Scanning images...")
for img_name in os.listdir(unlabeled_folder):
    if img_name.endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(unlabeled_folder, img_name)
        img = Image.open(img_path).convert('RGB')
        
        # Prepare image for model
        input_tensor = inference_transforms(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(input_tensor)
            # Convert raw output to probabilities (0.0 to 1.0)
            probabilities = torch.nn.functional.softmax(output[0], dim=0)
            
            # 'tripos' is index 1
            tripos_conf = probabilities[1].item() 
            
            if tripos_conf > 0.75:  # Adjust this threshold based on your needs
                shutil.copy(img_path, os.path.join(destination_folder, img_name))
                print(f"Found potential Tripos: {img_name} ({tripos_conf:.2%})")

print("Cleaning complete!")

Scanning images...
Found potential Tripos: 798.png (84.86%)
Found potential Tripos: 589.png (76.92%)
Found potential Tripos: 4242.png (75.61%)
Found potential Tripos: 2144.png (84.07%)
Found potential Tripos: 1466.png (82.32%)
Cleaning complete!
